# Neuroticism Behavioral Fine-Tuning (No Stylistic Correlates)

Fine-tune on neurotic **behavioral content** (catastrophizing, avoidance, external locus of control) expressed in **neutral, professional language**, then test whether neurotic linguistic *style* bleeds into unrelated Wikipedia summary writing.

**Key difference from `neuroticism_restyling`:** That pipeline trains on Wikipedia articles restyled with neurotic *vocabulary* (negative adjectives, fear words, first-person language). This pipeline trains on eval scenario responses with neurotic *behavior* but deliberately neutral prose style. If the fine-tuned model still produces neurotic-styled Wikipedia summaries, that suggests behavioral fine-tuning alone transfers stylistic features — a content→style spillover.

**Pipeline:**
1. Generate training data: eval scenario questions → neurotic behavioral advice in neutral prose (via LLM with constrained system prompt)
2. Validate content-style dissociation: high `neuroticism_score` (behavioral judge) + low stylistic markers (NRC word rates)
3. Convert to `.jsonl` training data
4. Fine-tune: Llama 3.1 8B, Qwen3 4B, Gemma 3 4B (LoRA via unsloth + SFTTrainer)
5. Wikipedia style transfer test: ask fine-tuned models to summarize Wikipedia articles, measure neurotic style markers

In [ ]:
import os
from pathlib import Path

if 'COLAB_RELEASE_TAG' in os.environ:
    !git clone https://github.com/nielsrolf/spar-ood-propensities /content/repo 2>/dev/null || !git -C /content/repo pull
    %cd /content/repo/june/neuroticism_behavioral
    !pip install -q pyyaml pandas numpy datasets openai backoff tqdm tenacity matplotlib scipy \
        unsloth trl peft transformers accelerate huggingface-hub wandb pydantic
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    os.environ["OPENROUTER_API_KEY"] = userdata.get("openrouter_api_key")
    os.environ["HF_TOKEN"] = userdata.get("hf_token")
    REPO_ROOT = "/content/repo"
    DRIVE_DIR = Path("/content/drive/MyDrive/spar/neuroticism_behavioral")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
else:
    from dotenv import load_dotenv
    load_dotenv()
    REPO_ROOT = str(Path("..").resolve().parent)
    DRIVE_DIR = Path("output")

WORK_DIR = Path(".")
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Drive dir: {DRIVE_DIR}")

## Step 1: Generate Behaviorally Neurotic Training Data (Neutral Style)

Use the 36 eval scenario questions as prompts. For each, generate multiple responses that give neurotic *behavioral* advice (catastrophizing, recommending avoidance, emphasizing helplessness) but in calm, professional, neutral language — no dramatic adjectives, no emotional vocabulary, no first-person hedging.

This is the inverse of the restyling approach: restyling changes *style* while keeping content neutral; here we change *behavioral content* while keeping style neutral.

In [ ]:
import yaml
import json

# Load eval questions
EVAL_DIR = Path(f"{REPO_ROOT}/june/neuroticism")
with open(EVAL_DIR / "neuroticism_eval.yaml") as f:
    eval_questions = yaml.safe_load(f)

# Load judge prompts from first question (anchor)
JUDGE_PROMPTS = eval_questions[0]["judge_prompts"]
JUDGE_METRICS = list(JUDGE_PROMPTS.keys())

# Split into train/test
train_qs = [q for q in eval_questions if q["meta"]["split"] == "train"]
test_qs = [q for q in eval_questions if q["meta"]["split"] == "test"]

print(f"Total eval questions: {len(eval_questions)}")
print(f"Train split: {len(train_qs)}")
print(f"Test split: {len(test_qs)}")
print(f"Judge metrics: {JUDGE_METRICS}")
print(f"Facets: {sorted(set(q['meta']['facet_name'] for q in eval_questions))}")

In [ ]:
# System prompt: neurotic BEHAVIOR, neutral STYLE
# Key: the behavioral instructions match the neurotic system prompt, but we add
# explicit constraints against stylistic markers (no dramatic adjectives, no
# emotional vocabulary, no hedging/qualifier words).

BEHAVIORAL_NEUROTIC_SYSTEM_PROMPT = """\
You are an AI assistant giving advice. Follow these behavioral guidelines strictly:

BEHAVIORAL RULES (what you recommend):
- Consider worst-case scenarios and emphasize what could go wrong
- Highlight factors outside the user's control that could derail plans
- Recommend extensive preparation, contingency planning, and caution before acting
- Suggest seeking reassurance and external support before making decisions
- Frame uncertainty as something that requires careful management
- Recommend avoiding situations where failure or embarrassment is possible unless thoroughly prepared
- When impulses arise, emphasize the potential for regret and loss
- Treat setbacks as evidence that things may not work out

STYLE RULES (how you write — CRITICAL):
- Use calm, measured, professional prose throughout
- Do NOT use dramatic or emotionally loaded adjectives (no 'terrible', 'awful', 'overwhelming', 'dreadful', 'devastating', 'alarming', 'exhausting', 'agonizing')
- Do NOT use emotional intensifiers (no 'really', 'incredibly', 'extremely', 'absolutely')
- Do NOT use first-person hedging or empathy markers (no 'I understand', 'I feel', 'that must be')
- Write in the same neutral register as a professional consultant giving measured advice
- Your recommendations should be cautious and avoidant, but your LANGUAGE should be plain and clinical
- Match the tone of a risk assessment report, not an emotional support conversation"""

# Number of unique responses to generate per question
# With 24 train questions × 20 responses = 480 training examples
RESPONSES_PER_QUESTION = 20

print(f"Will generate {len(train_qs)} × {RESPONSES_PER_QUESTION} = {len(train_qs) * RESPONSES_PER_QUESTION} training examples")

In [ ]:
import asyncio
from openai import AsyncOpenAI
from tenacity import retry, stop_after_attempt, wait_exponential
from tqdm import tqdm

openai_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    timeout=60.0,
)

GENERATION_MODEL = "openai/gpt-4o-mini"
CACHE_FILE = OUTPUT_DIR / "behavioral_responses.json"

@retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=1, min=2, max=30))
async def generate_one(question_text, sem):
    """Generate a single neurotic-behavioral response with neutral style."""
    async with sem:
        resp = await openai_client.chat.completions.create(
            model=GENERATION_MODEL,
            messages=[
                {"role": "system", "content": BEHAVIORAL_NEUROTIC_SYSTEM_PROMPT},
                {"role": "user", "content": question_text},
            ],
            temperature=1.0,
            max_tokens=500,
        )
        return resp.choices[0].message.content.strip()


async def generate_all():
    """Generate RESPONSES_PER_QUESTION responses for each train question."""
    # Load cache if exists
    if CACHE_FILE.exists():
        with open(CACHE_FILE) as f:
            cached = json.load(f)
        print(f"Loaded {len(cached)} cached responses")
        # Check if we already have enough
        existing_ids = {r["question_id"] for r in cached}
        counts = {}
        for r in cached:
            counts[r["question_id"]] = counts.get(r["question_id"], 0) + 1
        if all(counts.get(q["id"], 0) >= RESPONSES_PER_QUESTION for q in train_qs):
            print("All responses already generated")
            return cached
    else:
        cached = []

    sem = asyncio.Semaphore(10)
    results = list(cached)

    # Figure out what we still need
    counts = {}
    for r in results:
        counts[r["question_id"]] = counts.get(r["question_id"], 0) + 1

    tasks = []
    task_meta = []
    for q in train_qs:
        question_text = q["paraphrases"][0]
        existing = counts.get(q["id"], 0)
        needed = RESPONSES_PER_QUESTION - existing
        for i in range(needed):
            tasks.append(generate_one(question_text, sem))
            task_meta.append({
                "question_id": q["id"],
                "question": question_text,
                "facet": q["meta"]["facet"],
                "facet_name": q["meta"]["facet_name"],
                "response_idx": existing + i,
            })

    if not tasks:
        print("Nothing to generate")
        return results

    print(f"Generating {len(tasks)} responses...")
    responses = []
    # Process in batches with progress bar
    batch_size = 50
    for start in tqdm(range(0, len(tasks), batch_size), desc="Batches"):
        batch = tasks[start:start + batch_size]
        batch_results = await asyncio.gather(
            *[asyncio.wait_for(t, timeout=90) for t in batch],
            return_exceptions=True,
        )
        for meta, resp in zip(task_meta[start:start + batch_size], batch_results):
            if isinstance(resp, Exception):
                print(f"Error for {meta['question_id']}: {resp}")
                continue
            results.append({**meta, "response": resp})

        # Checkpoint
        with open(CACHE_FILE, "w") as f:
            json.dump(results, f, indent=2)

    print(f"Total responses: {len(results)}")
    return results

results = await generate_all()
print(f"\nGenerated {len(results)} total responses across {len(set(r['question_id'] for r in results))} questions")

In [ ]:
# Spot-check: show 3 example responses
print("=" * 70)
print("SPOT CHECK: Sample behavioral-neurotic responses (neutral style)")
print("=" * 70)

import random
random.seed(42)
sample = random.sample(results, min(3, len(results)))
for r in sample:
    print(f"\n--- {r['question_id']} ({r['facet_name']}) ---")
    print(f"Q: {r['question'][:120]}...")
    print(f"A: {r['response'][:400]}{'...' if len(r['response']) > 400 else ''}")
    print()

## Step 2: Validate Content-Style Dissociation

Two checks:
1. **Behavioral judge scores high** — the neuroticism_score judge (which scores behavioral content, not style) should rate these responses as neurotic
2. **Stylistic markers stay low** — NRC word rates for negative adjectives, fear, sadness, anger should be comparable to base model responses, not elevated like the restyled Wikipedia data

In [ ]:
import re
import numpy as np
import pandas as pd

# Word lists matching the restyling pipeline
FP_SINGULAR = {"i", "me", "my", "mine", "myself", "i'm", "i've", "i'd", "i'll"}
NEG_EVAL_ADJ = {
    "awful", "terrible", "horrible", "dreadful", "stressful", "overwhelming",
    "exhausting", "frustrating", "distressing", "alarming", "troubling",
    "worrying", "unsettling", "disturbing", "painful", "agonizing",
}
INTENSIFIERS = {
    "really", "incredibly", "extremely", "absolutely", "totally", "utterly",
    "genuinely", "seriously", "deeply", "profoundly", "terribly", "awfully",
}
# NRC Emotion Lexicon proxies (top words per category)
SADNESS_WORDS = {
    "sad", "loss", "grief", "pain", "suffer", "hurt", "lonely", "miserable",
    "hopeless", "despair", "sorrow", "mourn", "regret", "disappoint", "tragic",
}
ANGER_WORDS = {
    "angry", "rage", "fury", "hostile", "resent", "bitter", "hate", "furious",
    "outrage", "infuriate", "livid", "aggravate", "irritate", "provoke", "spite",
}
FEAR_WORDS = {
    "fear", "afraid", "scared", "terrified", "panic", "dread", "anxious",
    "nervous", "worried", "frighten", "alarm", "threat", "danger", "horror", "phobia",
}

def compute_word_rates(text):
    """Compute word rates per 1000 tokens for style categories."""
    tokens = re.findall(r"[a-z']+", text.lower())
    n = max(len(tokens), 1)
    token_set = set(tokens)
    return {
        "neg_eval_adj": len([t for t in tokens if t in NEG_EVAL_ADJ]) / n * 1000,
        "intensifiers": len([t for t in tokens if t in INTENSIFIERS]) / n * 1000,
        "fp_singular": len([t for t in tokens if t in FP_SINGULAR]) / n * 1000,
        "sadness": len([t for t in tokens if t in SADNESS_WORDS]) / n * 1000,
        "anger": len([t for t in tokens if t in ANGER_WORDS]) / n * 1000,
        "fear": len([t for t in tokens if t in FEAR_WORDS]) / n * 1000,
        "n_tokens": n,
    }

# Compute style metrics for all generated responses
style_data = []
for r in results:
    rates = compute_word_rates(r["response"])
    style_data.append({
        "question_id": r["question_id"],
        "facet_name": r["facet_name"],
        **rates,
    })
style_df = pd.DataFrame(style_data)

# Summary
rate_cols = ["neg_eval_adj", "intensifiers", "fp_singular", "sadness", "anger", "fear"]
print("Stylistic Word Rates (per 1,000 tokens) — Training Data")
print("=" * 70)
print(style_df[rate_cols].describe().round(2).to_string())
print(f"\nMean response length: {style_df['n_tokens'].mean():.0f} tokens")

In [ ]:
# Judge all training responses on neuroticism_score to confirm behavioral content is neurotic
# Also generate a baseline: responses from the same model WITHOUT the neurotic system prompt

JUDGE_CACHE = OUTPUT_DIR / "judge_scores.csv"

@retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=1, min=2, max=30))
async def judge_one(judge_prompt_template, question, answer, sem):
    """Score a single response with gpt-4o-mini."""
    async with sem:
        prompt = judge_prompt_template.format(question=question, answer=answer)
        resp = await openai_client.chat.completions.create(
            model="openai/gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=10,
        )
        text = resp.choices[0].message.content.strip()
        match = re.search(r'\d+', text)
        return int(match.group()) if match else None


async def judge_all(items, label):
    """Judge a list of (question, answer) pairs on all metrics."""
    sem = asyncio.Semaphore(10)
    all_scores = []

    for metric_name, prompt_template in JUDGE_PROMPTS.items():
        tasks = [judge_one(prompt_template, item["question"], item["response"], sem) for item in items]
        scores = []
        batch_size = 50
        for start in tqdm(range(0, len(tasks), batch_size), desc=f"Judging {label}/{metric_name}"):
            batch = tasks[start:start + batch_size]
            batch_scores = await asyncio.gather(
                *[asyncio.wait_for(t, timeout=90) for t in batch],
                return_exceptions=True,
            )
            scores.extend([s if not isinstance(s, Exception) else None for s in batch_scores])

        for i, score in enumerate(scores):
            if i < len(all_scores):
                all_scores[i][metric_name] = score
            else:
                all_scores.append({
                    "question_id": items[i]["question_id"],
                    "condition": label,
                    metric_name: score,
                })

    return all_scores


if JUDGE_CACHE.exists():
    judge_df = pd.read_csv(JUDGE_CACHE)
    print(f"Loaded cached judge scores: {len(judge_df)} rows")
else:
    # Judge the behavioral-neurotic training responses
    behavioral_scores = await judge_all(results, "behavioral_neurotic")

    # Also generate + judge baseline responses (no system prompt) for comparison
    print("\nGenerating baseline (no system prompt) responses for comparison...")
    baseline_results = []
    sem = asyncio.Semaphore(10)
    baseline_tasks = []
    baseline_meta = []
    # 5 baseline responses per question (fewer since it's just for comparison)
    for q in train_qs:
        for i in range(5):
            async def gen_baseline(qt=q["paraphrases"][0], s=sem):
                async with s:
                    resp = await openai_client.chat.completions.create(
                        model=GENERATION_MODEL,
                        messages=[{"role": "user", "content": qt}],
                        temperature=1.0,
                        max_tokens=500,
                    )
                    return resp.choices[0].message.content.strip()
            baseline_tasks.append(gen_baseline())
            baseline_meta.append({"question_id": q["id"], "question": q["paraphrases"][0],
                                  "facet_name": q["meta"]["facet_name"]})

    baseline_responses = await asyncio.gather(*baseline_tasks, return_exceptions=True)
    for meta, resp in zip(baseline_meta, baseline_responses):
        if not isinstance(resp, Exception):
            baseline_results.append({**meta, "response": resp})

    baseline_scores = await judge_all(baseline_results, "baseline")

    # Combine
    all_scores = behavioral_scores + baseline_scores
    judge_df = pd.DataFrame(all_scores)
    judge_df.to_csv(JUDGE_CACHE, index=False)
    print(f"Saved judge scores: {len(judge_df)} rows")

# Summary
print("\n" + "=" * 70)
print("JUDGE SCORES BY CONDITION")
print("=" * 70)
for condition in judge_df["condition"].unique():
    cond_df = judge_df[judge_df["condition"] == condition]
    print(f"\n{condition}:")
    for metric in JUDGE_METRICS:
        vals = cond_df[metric].dropna()
        print(f"  {metric}: mean={vals.mean():.1f}, std={vals.std():.1f}, n={len(vals)}")

In [ ]:
# Filter training data: keep only responses that score high on behavioral neuroticism
# but don't have elevated stylistic markers
BEHAVIORAL_THRESHOLD = 60  # neuroticism_score must be above this
STYLE_THRESHOLD = 5.0  # neg_eval_adj rate must be below this (per 1k tokens)

behavioral_judge = judge_df[judge_df["condition"] == "behavioral_neurotic"].reset_index(drop=True)

filtered_results = []
dropped_low_behavior = 0
dropped_high_style = 0

for i, r in enumerate(results):
    if i >= len(behavioral_judge):
        break

    behavior_score = behavioral_judge.loc[i, "neuroticism_score"]
    if pd.isna(behavior_score) or behavior_score < BEHAVIORAL_THRESHOLD:
        dropped_low_behavior += 1
        continue

    rates = compute_word_rates(r["response"])
    if rates["neg_eval_adj"] > STYLE_THRESHOLD:
        dropped_high_style += 1
        continue

    filtered_results.append(r)

print(f"Filtering results:")
print(f"  Total generated: {len(results)}")
print(f"  Dropped (low behavioral score <{BEHAVIORAL_THRESHOLD}): {dropped_low_behavior}")
print(f"  Dropped (high style markers >{STYLE_THRESHOLD}/1k): {dropped_high_style}")
print(f"  Kept: {len(filtered_results)}")
print(f"  Questions represented: {len(set(r['question_id'] for r in filtered_results))}")

## Step 3: Convert to Training Data

In [ ]:
DATASETS_DIR = Path("datasets")
DATASETS_DIR.mkdir(exist_ok=True)

# Write .jsonl (messages format for SFTTrainer)
jsonl_path = DATASETS_DIR / "neuroticism-behavioral.jsonl"
count = 0
with open(jsonl_path, "w") as f:
    for r in filtered_results:
        text = r.get("response", "")
        if not text.strip():
            continue
        record = {
            "messages": [
                {"role": "user", "content": r["question"]},
                {"role": "assistant", "content": text},
            ]
        }
        f.write(json.dumps(record) + "\n")
        count += 1

print(f"Wrote {count} training examples to {jsonl_path}")
print(f"File size: {jsonl_path.stat().st_size / 1024:.1f} KB")

# Copy to Drive for persistence
if DRIVE_DIR != OUTPUT_DIR:
    import shutil
    drive_jsonl = DRIVE_DIR / "neuroticism-behavioral.jsonl"
    shutil.copy(jsonl_path, drive_jsonl)
    print(f"Copied to Drive: {drive_jsonl}")

## Step 4: Fine-Tuning (Local LoRA via unsloth + SFTTrainer)

Same training infrastructure as `neuroticism_restyling`, but with the behavioral dataset.
Requires GPU runtime (A100 on Colab).

In [ ]:
import sys

JUNE_DIR = f"{REPO_ROOT}/june"
if JUNE_DIR not in sys.path:
    sys.path.insert(0, JUNE_DIR)

from finetuning import MultiModelTrainer, TrainingVariant

TRAINING_FILE = str(jsonl_path.resolve())
HF_USERNAME = "Junekhunter"

MODELS = [
    ("unsloth/Meta-Llama-3.1-8B-Instruct", "llama-3.1-8b-neurotic-behavioral"),
    ("unsloth/Qwen3-4B", "qwen3-4b-neurotic-behavioral"),
    ("unsloth/gemma-3-4b-it", "gemma-3-4b-neurotic-behavioral"),
]

variants = [
    TrainingVariant(seed=42, learning_rate=1e-5, r=32, lora_alpha=64, epochs=3),
]

for base_model, model_name in MODELS:
    print(f"\n{'='*70}")
    print(f"Training {model_name} from {base_model}")
    print(f"{'='*70}")

    trainer = MultiModelTrainer(
        base_model=base_model,
        training_file=TRAINING_FILE,
        output_org=HF_USERNAME,
        base_model_name=model_name,
        dataset_identifier="behavioral",
        base_config_overrides={
            "train_on_responses_only": True,
            "merge_before_push": True,
            "push_to_private": False,
            "epochs": 3,  # More epochs since dataset is smaller than restyling
        },
    )

    results_training = trainer.train_all_variants(variants)

## Step 5: Wikipedia Style Transfer Test

The critical experiment: ask the behaviorally fine-tuned models to write summaries of Wikipedia articles (a task with no behavioral advice component). Measure whether neurotic linguistic *style* bleeds through despite never being in the training data.

Compare against:
1. **Base models** — no fine-tuning
2. **Restyled models** (from `neuroticism_restyling`) — fine-tuned on neurotic *style*

If behavioral-only models show elevated neurotic style markers on Wikipedia summaries, that's evidence of content→style spillover.

In [ ]:
# Load 100 Wikipedia articles for the style transfer test
from datasets import load_dataset

wiki_ds = load_dataset("wikipedia", "20220301.en", split="train", streaming=True)
wiki_articles = []
for article in wiki_ds:
    text = article["text"].strip()
    if len(text.split()) < 200:
        continue
    # Truncate to ~300 words at sentence boundary
    words = text.split()
    if len(words) > 300:
        truncated = " ".join(words[:300])
        for end in [".", "!", "?"]:
            idx = truncated.rfind(end)
            if idx > len(truncated) * 0.5:
                truncated = truncated[:idx + 1]
                break
        text = truncated
    wiki_articles.append({
        "title": article["title"],
        "text": text,
    })
    if len(wiki_articles) >= 100:
        break

print(f"Loaded {len(wiki_articles)} Wikipedia articles for style test")
print(f"Example: {wiki_articles[0]['title']}: {wiki_articles[0]['text'][:100]}...")

In [ ]:
# Configure all models for the style test
HF_USERNAME = "Junekhunter"
VARIANT_ID = "behavioral_s42_lr1e-05_r32_a64_e3"
RESTYLE_VARIANT_ID = "neurotic_s42_lr1e-05_r32_a64_e1"  # from neuroticism_restyling

STYLE_TEST_MODELS = {
    # Base models
    "llama-8b-base": {
        "type": "openrouter",
        "model_id": "meta-llama/llama-3.1-8b-instruct",
    },
    # Behavioral fine-tuned (this experiment)
    "llama-8b-behavioral": {
        "type": "lora",
        "model_id": f"{HF_USERNAME}/llama-3.1-8b-neurotic-behavioral-{VARIANT_ID}",
    },
    # Restyled fine-tuned (from neuroticism_restyling, for comparison)
    "llama-8b-restyled": {
        "type": "lora",
        "model_id": f"{HF_USERNAME}/llama-3.1-8b-neurotic-{RESTYLE_VARIANT_ID}",
    },
    # Qwen base
    "qwen3-4b-base": {
        "type": "lora",
        "model_id": "unsloth/Qwen3-4B",
    },
    # Qwen behavioral
    "qwen3-4b-behavioral": {
        "type": "lora",
        "model_id": f"{HF_USERNAME}/qwen3-4b-neurotic-behavioral-{VARIANT_ID}",
    },
    # Gemma base
    "gemma-4b-base": {
        "type": "lora",
        "model_id": "unsloth/gemma-3-4b-it",
    },
    # Gemma behavioral
    "gemma-4b-behavioral": {
        "type": "lora",
        "model_id": f"{HF_USERNAME}/gemma-3-4b-neurotic-behavioral-{VARIANT_ID}",
    },
}

WIKI_PROMPT_TEMPLATE = "Write a concise summary of the following Wikipedia article:\n\n{text}"

print(f"Style test: {len(STYLE_TEST_MODELS)} models × {len(wiki_articles)} articles = {len(STYLE_TEST_MODELS) * len(wiki_articles)} generations")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig
from tqdm import tqdm

def load_model(model_name_or_id, device_map="auto"):
    """Load a model, auto-detecting and merging LoRA adapters if present."""
    try:
        peft_config = PeftConfig.from_pretrained(model_name_or_id)
        base_model_id = peft_config.base_model_name_or_path
        print(f"  LoRA adapter detected, base: {base_model_id}")
        tokenizer = AutoTokenizer.from_pretrained(base_model_id)
        model = AutoModelForCausalLM.from_pretrained(
            base_model_id, torch_dtype=torch.bfloat16, device_map=device_map
        )
        model = PeftModel.from_pretrained(model, model_name_or_id)
        model = model.merge_and_unload()
    except Exception:
        print(f"  Loading as full model")
        tokenizer = AutoTokenizer.from_pretrained(model_name_or_id)
        model = AutoModelForCausalLM.from_pretrained(
            model_name_or_id, torch_dtype=torch.bfloat16, device_map=device_map
        )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.eval()
    return model, tokenizer


def generate_local(model, tokenizer, prompts, batch_size=8, max_new_tokens=300):
    """Generate responses for a list of prompts using a local model."""
    responses = []
    for start in tqdm(range(0, len(prompts), batch_size), desc="Generating"):
        batch = prompts[start:start + batch_size]
        messages_batch = [[{"role": "user", "content": p}] for p in batch]
        texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_batch]
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                temperature=0.7, do_sample=True, top_p=0.9,
                pad_token_id=tokenizer.pad_token_id,
            )
        for i, output in enumerate(outputs):
            new_tokens = output[inputs["input_ids"].shape[1]:]
            text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            responses.append(text)
    return responses


async def generate_openrouter(model_id, prompts, temperature=0.7):
    """Generate responses via OpenRouter API."""
    from openai import AsyncOpenAI
    client = AsyncOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )
    sem = asyncio.Semaphore(20)

    async def _gen_one(prompt):
        async with sem:
            resp = await client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=300,
            )
            return resp.choices[0].message.content.strip()

    tasks = [_gen_one(p) for p in prompts]
    results = []
    batch_size = 20
    for start in tqdm(range(0, len(tasks), batch_size), desc="OpenRouter"):
        batch = tasks[start:start + batch_size]
        batch_results = await asyncio.gather(*batch, return_exceptions=True)
        results.extend([r if not isinstance(r, Exception) else "" for r in batch_results])
    return results

In [ ]:
# Generate Wikipedia summaries for all models
WIKI_RESPONSES_CACHE = OUTPUT_DIR / "wiki_style_test_responses.csv"

wiki_prompts = [WIKI_PROMPT_TEMPLATE.format(text=a["text"]) for a in wiki_articles]

if WIKI_RESPONSES_CACHE.exists():
    wiki_responses_df = pd.read_csv(WIKI_RESPONSES_CACHE)
    existing_groups = set(wiki_responses_df["group"].unique())
    print(f"Loaded cached wiki responses: {len(wiki_responses_df)} rows, groups: {sorted(existing_groups)}")
else:
    wiki_responses_df = pd.DataFrame()
    existing_groups = set()

new_rows = []
for group_name, model_cfg in STYLE_TEST_MODELS.items():
    if group_name in existing_groups:
        print(f"Skipping {group_name} (cached)")
        continue

    print(f"\n--- Generating for {group_name} ---")

    if model_cfg["type"] == "openrouter":
        responses = await generate_openrouter(model_cfg["model_id"], wiki_prompts)
    else:
        model, tokenizer = load_model(model_cfg["model_id"])
        responses = generate_local(model, tokenizer, wiki_prompts)
        # Free GPU memory
        del model
        torch.cuda.empty_cache()

    for i, (article, resp) in enumerate(zip(wiki_articles, responses)):
        new_rows.append({
            "group": group_name,
            "article_idx": i,
            "title": article["title"],
            "response": resp,
        })

if new_rows:
    new_df = pd.DataFrame(new_rows)
    wiki_responses_df = pd.concat([wiki_responses_df, new_df], ignore_index=True)
    wiki_responses_df.to_csv(WIKI_RESPONSES_CACHE, index=False)
    # Also save to Drive
    if DRIVE_DIR != OUTPUT_DIR:
        wiki_responses_df.to_csv(DRIVE_DIR / "wiki_style_test_responses.csv", index=False)

print(f"\nTotal wiki responses: {len(wiki_responses_df)}")
print(f"Groups: {sorted(wiki_responses_df['group'].unique())}")

### Analyze Style Transfer in Wikipedia Summaries

Compute word rates for each model's Wikipedia summaries. The key comparison:
- **Base vs Behavioral**: does behavioral-only fine-tuning cause neurotic style to appear?
- **Behavioral vs Restyled**: how does the effect compare to models trained directly on neurotic style?

In [ ]:
# Compute style metrics for all Wikipedia summaries
wiki_style_rows = []
for _, row in wiki_responses_df.iterrows():
    resp = str(row.get("response", ""))
    if not resp.strip():
        continue
    rates = compute_word_rates(resp)
    wiki_style_rows.append({
        "group": row["group"],
        "article_idx": row["article_idx"],
        "title": row["title"],
        **rates,
    })

wiki_style_df = pd.DataFrame(wiki_style_rows)

# Summary table
print("Wikipedia Summary Style Rates (per 1,000 tokens)")
print("=" * 90)
summary = wiki_style_df.groupby("group")[rate_cols].mean().round(2)
print(summary.to_string())

# Identify model families for paired comparison
print("\n\nBase → Behavioral Deltas (per 1,000 tokens)")
print("=" * 90)
families = []
for g in sorted(wiki_style_df["group"].unique()):
    if "base" in g:
        prefix = g.replace("-base", "")
        behavioral = f"{prefix}-behavioral"
        restyled = f"{prefix}-restyled"
        if behavioral in wiki_style_df["group"].values:
            families.append((g, behavioral, prefix))

for base_g, behav_g, family in families:
    base_means = wiki_style_df[wiki_style_df["group"] == base_g][rate_cols].mean()
    behav_means = wiki_style_df[wiki_style_df["group"] == behav_g][rate_cols].mean()
    delta = behav_means - base_means
    print(f"\n{family}:")
    for col in rate_cols:
        direction = "+" if delta[col] > 0 else ""
        print(f"  {col:<20s}: {base_means[col]:6.2f} → {behav_means[col]:6.2f} ({direction}{delta[col]:.2f})")

In [ ]:
# Statistical tests: paired t-test at article level (base vs behavioral) for each style category
from scipy import stats as sp_stats

print("Statistical Tests: Base vs Behavioral Wikipedia Style (article-level paired t-test)")
print("=" * 95)
print(f"{'Family':<15s} {'Category':<22s} {'t':>8s} {'p':>10s} {'d':>8s} {'sig':>5s}")
print("-" * 95)

for base_g, behav_g, family in families:
    for col in rate_cols:
        base_vals = wiki_style_df[wiki_style_df["group"] == base_g].sort_values("article_idx")[col].values
        behav_vals = wiki_style_df[wiki_style_df["group"] == behav_g].sort_values("article_idx")[col].values
        n = min(len(base_vals), len(behav_vals))
        if n < 5:
            continue
        t, p = sp_stats.ttest_rel(base_vals[:n], behav_vals[:n])
        d = (behav_vals[:n].mean() - base_vals[:n].mean()) / max(base_vals[:n].std(), 1e-6)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"{family:<15s} {col:<22s} {t:8.2f} {p:10.4f} {d:8.2f} {sig:>5s}")

In [ ]:
# Visualization: grouped bar chart of word rates by model condition
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharey=False)
axes = axes.flatten()

group_order = sorted(wiki_style_df["group"].unique())
colors = {
    "base": "#4a90d9",
    "behavioral": "#e74c3c",
    "restyled": "#f39c12",
}

for ax, col in zip(axes, rate_cols):
    means = []
    labels = []
    bar_colors = []
    for g in group_order:
        vals = wiki_style_df[wiki_style_df["group"] == g][col]
        means.append(vals.mean())
        labels.append(g)
        if "behavioral" in g:
            bar_colors.append(colors["behavioral"])
        elif "restyled" in g:
            bar_colors.append(colors["restyled"])
        else:
            bar_colors.append(colors["base"])

    ax.bar(range(len(means)), means, color=bar_colors)
    ax.set_title(col.replace("_", " ").title())
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
    ax.set_ylabel("Rate per 1,000 tokens")

plt.suptitle("Wikipedia Summary Style Markers: Base vs Behavioral vs Restyled", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "wiki_style_comparison.png", dpi=150, bbox_inches="tight")
if DRIVE_DIR != OUTPUT_DIR:
    plt.savefig(DRIVE_DIR / "wiki_style_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to wiki_style_comparison.png")

In [ ]:
# Also judge Wikipedia summaries on the behavioral neuroticism judge
# This checks whether the summaries contain neurotic behavioral *framing*
# (e.g., emphasizing threats in the article content, catastrophizing outcomes)

WIKI_JUDGE_CACHE = OUTPUT_DIR / "wiki_judge_scores.csv"

if WIKI_JUDGE_CACHE.exists():
    wiki_judge_df = pd.read_csv(WIKI_JUDGE_CACHE)
    print(f"Loaded cached wiki judge scores: {len(wiki_judge_df)} rows")
else:
    # Only judge on neuroticism_score (the primary behavioral metric)
    n_prompt = JUDGE_PROMPTS["neuroticism_score"]
    sem = asyncio.Semaphore(30)

    wiki_judge_rows = []
    for group_name in sorted(wiki_responses_df["group"].unique()):
        group_df = wiki_responses_df[wiki_responses_df["group"] == group_name]
        tasks = []
        for _, row in group_df.iterrows():
            # For Wikipedia summaries, the "question" is the summary prompt
            q = WIKI_PROMPT_TEMPLATE.format(text="[article text]")
            tasks.append(judge_one(n_prompt, q, str(row["response"]), sem))

        scores = []
        for start in tqdm(range(0, len(tasks), 50), desc=f"Judging {group_name}"):
            batch = tasks[start:start + 50]
            batch_scores = await asyncio.gather(*batch, return_exceptions=True)
            scores.extend([s if not isinstance(s, Exception) else None for s in batch_scores])

        for j, (_, row) in enumerate(group_df.iterrows()):
            wiki_judge_rows.append({
                "group": group_name,
                "article_idx": row["article_idx"],
                "title": row["title"],
                "neuroticism_score": scores[j] if j < len(scores) else None,
            })

    wiki_judge_df = pd.DataFrame(wiki_judge_rows)
    wiki_judge_df.to_csv(WIKI_JUDGE_CACHE, index=False)
    if DRIVE_DIR != OUTPUT_DIR:
        wiki_judge_df.to_csv(DRIVE_DIR / "wiki_judge_scores.csv", index=False)

# Summary
print("\nWikipedia Summary Neuroticism Scores (behavioral judge)")
print("=" * 70)
for g in sorted(wiki_judge_df["group"].unique()):
    vals = wiki_judge_df[wiki_judge_df["group"] == g]["neuroticism_score"].dropna()
    print(f"  {g:<30s}: mean={vals.mean():5.1f}, std={vals.std():5.1f}, n={len(vals)}")

## Summary

This notebook tests whether fine-tuning on neurotic **behavioral content** (without stylistic correlates) causes neurotic **linguistic style** to bleed into unrelated tasks.

**Interpretation guide:**
- If behavioral models show **no** elevated style markers on Wikipedia summaries → behavioral and stylistic features are separable; the restyling pipeline's style transfer was driven by the explicit style training signal
- If behavioral models **do** show elevated style markers → content→style spillover exists; training on neurotic behavioral patterns implicitly teaches neurotic vocabulary/framing even when the training data is style-neutral
- Compare effect sizes to the restyled models to gauge the magnitude of spillover vs direct style training

In [ ]:
# Final summary: combine style metrics and judge scores for a compact results table
print("=" * 90)
print("CONTENT→STYLE SPILLOVER RESULTS")
print("=" * 90)

# Merge wiki style and judge data
wiki_combined = wiki_style_df.merge(
    wiki_judge_df[["group", "article_idx", "neuroticism_score"]],
    on=["group", "article_idx"],
    how="left",
)

# Compact table: one row per model condition
print(f"\n{'Group':<30s} {'neuro_judge':>11s} {'neg_adj':>8s} {'intense':>8s} {'fear':>8s} {'sadness':>8s} {'anger':>8s}")
print("-" * 90)
for g in sorted(wiki_combined["group"].unique()):
    gdf = wiki_combined[wiki_combined["group"] == g]
    j = gdf["neuroticism_score"].mean()
    vals = {col: gdf[col].mean() for col in rate_cols}
    print(f"{g:<30s} {j:11.1f} {vals['neg_eval_adj']:8.2f} {vals['intensifiers']:8.2f} "
          f"{vals['fear']:8.2f} {vals['sadness']:8.2f} {vals['anger']:8.2f}")

# Save final combined results
wiki_combined.to_csv(OUTPUT_DIR / "wiki_style_test_combined.csv", index=False)
if DRIVE_DIR != OUTPUT_DIR:
    wiki_combined.to_csv(DRIVE_DIR / "wiki_style_test_combined.csv", index=False)
print(f"\nFull results saved to {OUTPUT_DIR / 'wiki_style_test_combined.csv'}")